# Silver Cleaning

## Imports

In [17]:
from pyspark.sql.functions import (
    col,
    trim,
    substring,
    to_date,
    when,
    sum as spark_sum,
    current_timestamp,
    lit,
    lower,
    upper
)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 19, Finished, Available, Finished, False)

## Read Bronze Table

In [2]:
bronze_df = spark.sql("SELECT * FROM bronze.complaints")

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 4, Finished, Available, Finished, False)

In [3]:
display(bronze_df.limit(10))

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 684f6777-c352-46da-b3bd-bd0a3b9e2d33)

In [4]:
bronze_df.printSchema()

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 6, Finished, Available, Finished, False)

root
 |-- date_received: string (nullable = true)
 |-- product: string (nullable = true)
 |-- sub_product: string (nullable = true)
 |-- issue: string (nullable = true)
 |-- sub_issue: string (nullable = true)
 |-- consumer_complaint_narrative: string (nullable = true)
 |-- company_public_response: string (nullable = true)
 |-- company: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- date_sent_to_company: string (nullable = true)
 |-- company_response_to_consumer: string (nullable = true)
 |-- timely_response: string (nullable = true)
 |-- complaint_id: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)



## Remove Duplicate Rows

In [5]:
rows_before_dedup = bronze_df.count()

silver_deduped_df = bronze_df.dropDuplicates()

rows_after_dedup = silver_deduped_df.count()

duplicate_rows_removed = rows_before_dedup - rows_after_dedup
duplicate_rows_removed_pct = (duplicate_rows_removed / rows_before_dedup) * 100
XXXXX
print(f"Rows before deduplication: {rows_before_dedup:,}")
print(f"Rows after deduplication: {rows_after_dedup:,}")
print(f"Rows removed: {duplicate_rows_removed:,}")
print(f"Percentage removed: {duplicate_rows_removed_pct:.6f}%")

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 7, Finished, Available, Finished, False)

Rows before deduplication: 16,294,931
Rows after deduplication: 16,294,807
Rows removed: 124
Percentage removed: 0.000761%


## Validate Required Fields

In [6]:
missing_required_fields_condition = (
    col("complaint_id").isNull() |
    (trim(col("complaint_id")) == "") |
    col("date_received").isNull() |
    (trim(col("date_received")) == "") |
    col("product").isNull() |
    (trim(col("product")) == "") |
    col("issue").isNull() |
    (trim(col("issue")) == "") |
    col("company").isNull() |
    (trim(col("company")) == "")
)

missing_required_fields_df = silver_deduped_df.filter(missing_required_fields_condition)
valid_required_fields_df = silver_deduped_df.filter(~missing_required_fields_condition)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 8, Finished, Available, Finished, False)

In [7]:
missing_required_fields_count = missing_required_fields_df.count()
valid_required_fields_count = valid_required_fields_df.count()

print(f"Rows missing required fields: {missing_required_fields_count:,}")
print(f"Rows having required fields: {valid_required_fields_count:,}")

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 9, Finished, Available, Finished, False)

Rows missing required fields: 8,505
Rows having required fields: 16,286,302


In [8]:
display(
    missing_required_fields_df.select(
        "complaint_id",
        "date_received",
        "product",
        "issue",
        "company"
    ).limit(20)
)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a4b33380-0c98-4bcf-87d2-19d3450f7a3b)

- Rows missing complaint_id, date_received, product, issue, or company were separated from the main Silver DataFrame.
- These records will be kept for quarantine review instead of being dropped.

## Parse Date Fields

In [9]:
silver_dates_df = (
    valid_required_fields_df
    .withColumn(
        "date_received_clean",
        to_date(substring(col("date_received"), 1, 10), "yyyy-MM-dd")
    )
    .withColumn(
        "date_sent_to_company_clean",
        to_date(substring(col("date_sent_to_company"), 1, 10), "yyyy-MM-dd")
    )
)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 11, Finished, Available, Finished, False)

In [10]:
display(
    silver_dates_df.select(
        "date_received",
        "date_received_clean",
        "date_sent_to_company",
        "date_sent_to_company_clean"
    ).limit(20)
)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ed2eb81b-c0ef-464b-91b5-514d419c407d)

## Validate Date Fields

In [11]:
received_date_missing = (
    col("date_received").isNull()
    | (trim(col("date_received")) == "")
)

received_date_failed_to_parse = (
    col("date_received").isNotNull()
    & (trim(col("date_received")) != "")
    & col("date_received_clean").isNull()
)

sent_date_missing = (
    col("date_sent_to_company").isNull()
    | (trim(col("date_sent_to_company")) == "")
)

sent_date_failed_to_parse = (
    col("date_sent_to_company").isNotNull()
    & (trim(col("date_sent_to_company")) != "")
    & col("date_sent_to_company_clean").isNull()
)

sent_date_before_received_date = (
    col("date_received_clean").isNotNull()
    & col("date_sent_to_company_clean").isNotNull()
    & (col("date_sent_to_company_clean") < col("date_received_clean"))
)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 13, Finished, Available, Finished, False)

In [12]:
invalid_date_condition = (
    received_date_missing
    | received_date_failed_to_parse
    | sent_date_failed_to_parse
    | sent_date_before_received_date
)

invalid_date_df = silver_dates_df.filter(invalid_date_condition)

valid_date_df = silver_dates_df.filter(~invalid_date_condition)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 14, Finished, Available, Finished, False)

In [13]:
display(
    silver_dates_df.agg(
        spark_sum(when(received_date_missing, 1).otherwise(0)).alias("missing_date_received"),
        spark_sum(when(received_date_failed_to_parse, 1).otherwise(0)).alias("received_date_failed_to_parse"),
        spark_sum(when(sent_date_missing, 1).otherwise(0)).alias("missing_date_sent_to_company"),
        spark_sum(when(sent_date_failed_to_parse, 1).otherwise(0)).alias("sent_date_failed_to_parse"),
        spark_sum(when(sent_date_before_received_date, 1).otherwise(0)).alias("sent_date_before_received_date")
    )
)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f48d2b7d-d8cd-4be8-85fd-45e1f2e0c0da)

In [14]:
missing_sent_date_count = silver_dates_df.filter(sent_date_missing).count()

invalid_date_count = invalid_date_df.count()
valid_date_count = valid_date_df.count()

date_related_findings_count = missing_sent_date_count + invalid_date_count

print(f"Rows with missing date_sent_to_company: {missing_sent_date_count:,}")
print(f"Rows with invalid date logic: {invalid_date_count:,}")
print(f"Total date related findings: {date_related_findings_count:,}")
print(f"Rows with valid dates: {valid_date_count:,}")

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 16, Finished, Available, Finished, False)

Rows with missing date_sent_to_company: 26,215
Rows with invalid date logic: 7,050
Total date related findings: 33,265
Rows with valid dates: 16,279,252


- Some records did not have a date_sent_to_company value, but I kept them because they can still be used for complaint analysis.
- I separated the records where the sent-to-company date was earlier than the received date, since that date order does not make sense.

## Standardize Location Fields

In [28]:
zip_text = trim(col("zip_code"))

missing_zip = (
    col("zip_code").isNull()
    | (zip_text == "")
    | (lower(zip_text) == "unknown")
)

valid_zip = zip_text.rlike("^[0-9]{5}")

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 30, Finished, Available, Finished, False)

In [29]:
silver_location_df = (
    valid_date_df
    .withColumn(
        "state_clean",
        when(
            col("state").isNull() | (trim(col("state")) == ""),
            lit(None)
        ).otherwise(upper(trim(col("state"))))
    )
    .withColumn(
        "zip_code_clean",
        when(missing_zip, lit(None))
        .when(valid_zip, substring(zip_text, 1, 5))
        .otherwise(lit(None))
    )
    .withColumn(
        "zip_code_status",
        when(missing_zip, "missing_zip")
        .when(valid_zip, "valid_zip")
        .otherwise("masked_zip")
    )
)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 31, Finished, Available, Finished, False)

In [30]:
display(
    silver_location_df
    .groupBy("zip_code_status")
    .count()
    .orderBy("zip_code_status")
)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 276d1c2b-3f32-45bb-8a9d-59a40565f49d)

## Prepare Final Silver DataFrame

In [31]:
silver_complaints_df = (
    silver_location_df
    .withColumn("silver_processed_timestamp", current_timestamp())
)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 33, Finished, Available, Finished, False)

In [32]:
display(silver_complaints_df.limit(10))

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 04243df9-c0d7-405c-abbe-e5c37179a2d5)

In [33]:
silver_complaints_df.printSchema()

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 35, Finished, Available, Finished, False)

root
 |-- date_received: string (nullable = true)
 |-- product: string (nullable = true)
 |-- sub_product: string (nullable = true)
 |-- issue: string (nullable = true)
 |-- sub_issue: string (nullable = true)
 |-- consumer_complaint_narrative: string (nullable = true)
 |-- company_public_response: string (nullable = true)
 |-- company: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- date_sent_to_company: string (nullable = true)
 |-- company_response_to_consumer: string (nullable = true)
 |-- timely_response: string (nullable = true)
 |-- complaint_id: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- date_received_clean: date (nullable = true)
 |-- date_sent_to_company_clean: date (nullable = true)
 |-- state_clean: string (nullabl

## Prepare Quarantine DataFrame

In [34]:
missing_required_quarantine_df = (
    missing_required_fields_df
    .withColumn("quarantine_reason", lit("missing_required_field"))
)

invalid_date_quarantine_df = (
    invalid_date_df
    .withColumn("quarantine_reason", lit("date_quality_issue"))
)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 36, Finished, Available, Finished, False)

In [35]:
quarantine_complaints_df = missing_required_quarantine_df.unionByName(
    invalid_date_quarantine_df,
    allowMissingColumns=True
)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 37, Finished, Available, Finished, False)

In [36]:
display(
    quarantine_complaints_df.select(
        "complaint_id",
        "date_received",
        "date_received_clean",
        "date_sent_to_company",
        "date_sent_to_company_clean",
        "product",
        "issue",
        "company",
        "quarantine_reason"
    ).limit(20)
)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c13aa8f8-d699-4da5-9600-986a799c0558)

In [37]:
print(f"Rows in silver_complaints: {silver_complaints_df.count():,}")
print(f"Rows in quarantine_complaints: {quarantine_complaints_df.count():,}")

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 39, Finished, Available, Finished, False)

Rows in silver_complaints: 16,279,252
Rows in quarantine_complaints: 15,555


## Create Output Schemas

In [38]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS quarantine")

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 40, Finished, Available, Finished, False)

DataFrame[]

## Write Silver Tables

In [39]:
(
    silver_complaints_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.complaints")
)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 41, Finished, Available, Finished, False)

In [40]:
(
    quarantine_complaints_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("quarantine.complaints")
)

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 42, Finished, Available, Finished, False)

In [41]:
display(spark.sql("SELECT * FROM silver.complaints LIMIT 10"))

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6eac12da-a271-46a3-a0ab-d3a11a67c0be)

In [42]:
display(spark.sql("SELECT * FROM quarantine.complaints LIMIT 10"))

StatementMeta(, a47182de-b111-43cc-9672-2ac486325bc0, 44, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 33c4ab2d-4485-448a-86d3-f078d7d1aca7)